# LLM Workflows
This notebook accompanies the LLM Workflows slides for Comp 255.

1. [Pre-trained models](#pre-trained-models)
    - [Exercise - what is missing?](#exercise---what-is-missing)
2. [Instruction tuned models](#instruction-tuned-models)
    - [Exercise - experimentation with parameters](#exercise---experimentation-with-parameters)
    - [Exercise - what ELSE is missing?](#exercise---what-else-is-missing)
3. [Memory](#memory)
4. [Append-based memory](#append-based-memory)
5. [Tools](#tools)
6. [Structured outputs](#structured-outputs)
7. [Healthbot workflow](#healthbot-workflow)
8. [Healthbot workflow (alternate): functions](#healthbot-workflow-alternate-functions)

In [ ]:
# interacting with ollama using two different methods (see below)
import requests
from datetime import date
from ollama import chat, generate
from pydantic import BaseModel, Field
from typing import Literal

pretrained_model = "llama3.2:1b-text-q5_K_S"
sft_model = "llama3.2:1b"

## Pre-trained models
Right now we're using a simple "pre-trained" version of Llama.  It's just been trained on the language modeling objective; it learns to predict the next word.  As a result, you can see the output just continues the input text.

In [ ]:
response = generate(
    model=pretrained_model,
    prompt="What is the capital of France?",
    options={"num_predict": 50},
)
print(response.response)

### Exercise - what is missing?
You've probably used ChatGPT before.  What is missing from this bot? Experiment with having a conversation and note down what is missing.

* Does the bot answer your questions?
* What does the bot seem to be doing? Think about how this model is trained
* Does the bot seem to recall what it previously said to you?

In [ ]:
# asking a question
response = generate(
    model=pretrained_model,
    prompt="Hello, how are you?",
    options={"num_predict": 50},
)
print(response.response)
print("\n---\n")
# memory
response = generate(
    model=pretrained_model,
    prompt="What did you just say?",
    options={"num_predict": 50},
)
print(response.response)
print("\n---\n")
# continuation
response = generate(
    model=pretrained_model,
    prompt="Once upon a time, there was a brave knight who",
    options={"num_predict": 50},
)
print(response.response)

## Instruction-tuned models
Usually the above won't give us useful answers.  That's because the model is not tuned to produce useful answers, just to predict the next word.

That's where instruction tuning comes in.  That's covered in the slides, but here we'll re-run some of the above with a model that has been instruction tuned (Qwen2.5).

In this case we're using ollama's `chat` function, which formats the input to the model differently than `generate`.  We won't get into the details here, but you can take a look [details on templates](https://github.com/ollama/ollama/blob/main/docs/template.mdx) for more information.

In [ ]:
response = chat(
    model=sft_model,
    messages=[{"role": "user", "content": "What is the capital of France?"}],
)
print(response.message.content)

## Memory

One key feature of ChatGPT (and other chatbots) is the ability to remember previous interactions.  This is not only important for having interactions appear "conversational", but also to carry forward information e.g. from the output of a function.

Let's see what happens if we try that with our current setup.


In [ ]:
response = chat(
    model=sft_model,
    messages=[{"role": "user", "content": "My name is Ben."}],
)
print(response.message.content)
print("\n---\n")

response = chat(
    model=sft_model,
    messages=[{"role": "user", "content": "What is my name?"}],
)
print(response.message.content)

## Append-based memory

The simplest possible way to add "memory" is to append each turn to a message list and pass that full history into `chat`.  Let's implement that and see if it works.

In [ ]:
conversation_history = []

input_1 = {"role": "user", "content": "My name is Ben."}
conversation_history.append(input_1)

response_1 = chat(model=sft_model, messages=conversation_history)
print(response_1.message.content)
print("\n---\n")

conversation_history.append({"role": "assistant", "content": response_1.message.content})
input_2 = {"role": "user", "content": "What is my name?"}
conversation_history.append(input_2)
print('Conversation history:\n', '\n'.join(
    [f'{m["role"]}: {m["content"]}' for m in conversation_history]))
print("\n---\n")

response_2 = chat(model=sft_model, messages=conversation_history)
print(response_2.message.content)

## Tools
Our tool is simple - just get today's date.  Let's see what the model sees.

You'll notice sometimes this is brittle (i.e. it will give nonsense tool calls, it will ignore the tool results).  We're using an absolutely tiny model - <1% of the size of a foundation model.  The "amazing" things AI can do tends to require Very LLMs.


In [ ]:
# notice the detailed docstrings - this is to allow the LLM to understand its purpose
# we also are giving it a parameter, it really likes having parameters
def today(text: str) -> str:
    """Returns today's date, use this when you need to get today's date.\
    The input should always be an empty string."""
    return str(date.today())

# this will be useful later
available_tool_dict = {'today': today}

In [ ]:
# initialize memory
conversation_history = []

input_1 = {'role': 'user', 'content': 'What is the date today?'}
conversation_history.append(input_1)

response = chat(model=sft_model, messages=conversation_history,
  tools=[today], 
  # this makes our output more deterministic
  options={"temperature": 0},
)

# the response is a tool call! but we'll need a way to execute it
print(response.message)

# append the tool call to the conversation history
conversation_history.append(response.message)

The model sees:

```
<|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023

When you receive a tool call response, use the output to format an answer to the orginal user question.

You are a helpful assistant with tool calling capabilities.<|eot_id|><|start_header_id|>user<|end_header_id|>

Given the following functions, please respond with a JSON for a function call with its proper arguments that best answers the given prompt.

Respond in the format {"name": function name, "parameters": dictionary of argument name and its value}. Do not use variables.

{"type":"function","function":{"name":"today","description":"Returns today's date, use this when you need to get today's date.    The input should always be an empty string.","parameters":{"type":"object","required":["text"],"properties":{"text":{"type":"string"}}}}}

Question: What day is it today?<|eot_id|><|start_header_id|>assistant<|end_header_id|>
```

The `ollama` library post-processes the response to give us the `tool_calls` section of the response.  This can then be parsed and used to call a tool

In [ ]:
# process the output into a dictionary
tool_call = response.message.tool_calls[0].model_dump()['function']

if tool_call['name'] in available_tool_dict:
    tool_function = available_tool_dict[tool_call['name']]
    tool_args = tool_call['arguments']
    tool_result = tool_function(**tool_args)
    print(f"Tool result: {tool_result}")

Now we can just manually take that output and pass it back to the model.  In this case, we'll use the role "tool".  Again, this has to do with the templates with which the model was trained.

Note again - this will be brittle with a small model.  Just re-run it if it fails.

In [ ]:
tool_response = {'role': 'tool', 'content': tool_result}
response = chat(model=sft_model, 
    messages=conversation_history + [tool_response], 
    tools=[today])
response.message

## Structured outputs

One additional interesting thing we get out of the `ollama` library is that we can constrain what the model outputs to fit a particular format.  We can imagine this is useful when you want to impose some structure on outputs.  This uses a Pydantic data model, which is a legible way to describe the structure of a data object.

The library accomplishes this through something called [constrained decoding](https://www.aidancooper.co.uk/constrained-decoding/).  But that's a topic for when you have free time 😊

Quick example: take today's date string and parse it into month, date, and year.

In [ ]:
class ParsedDate(BaseModel):
    month: int
    date: int
    year: int

response = chat(
    model=sft_model,
    messages=[{
        "role": "user",
        "content": f"Parse this date into month, date, and year: {today('')}",
    }],
    format=ParsedDate.model_json_schema(),
    options={"temperature": 0}, # this makes the response faster
)

parsed = ParsedDate.model_validate_json(response.message.content)
print(response.message.content)
print(parsed)

## Healthbot workflow

Now let's build a sensible workflow.  See the slides for more details, but this is what we're aiming to build:

placholder for visual


In [ ]:
# pydantic models for constrained decoding

class Diagnosis(BaseModel):
    diagnosis: str = Field(description="The likely diagnosis")
    confidence: Literal["low", "medium", "high"] = Field(
        description="How confident we are in the diagnosis"
    )

class Symptoms(BaseModel):
    symptoms: list[str] = Field(description="A list of symptoms")   

class Record(BaseModel):
    patient_input: str
    symptoms: list[str]
    search_query: str | None = None
    search_results: str | None = None
    diagnosis: Diagnosis

In [ ]:
# defining our tool
def search_web(query: str) -> str:
    """Search the web for diagnosis information.

    Args:
        query: Query string to look up, e.g. "Diagnosis for cough, sneeze, runny nose".

    Returns:
        A string of the search results, e.g.
        - Title
        - Description
    """
    r = requests.post(
        "https://api.firecrawl.dev/v2/search",
        headers={"Content-Type": "application/json"},
        json={"query": query, "limit": 3},
        timeout=30,
    )
    r.raise_for_status()
    data = r.json()
    search_results = data['data']['web']
    lines = []
    for item in search_results:
        title = item.get("title", "")
        desc = (item.get("description") or "").replace("\n", " ")[:300]
        lines.append(f"- {title}\n {desc}")
    return "\n".join(lines)

# just a helper for calling tools
available_tool_dict = {'search_web': search_web}

# checking that this works:
search_web("Diagnosis for cough, sneeze, runny nose")

### Step functions

Each function does one thing from the slide diagram.  Quick smoke-tests after each definition.

In [ ]:
def parse_symptoms(patient_input: str) -> Symptoms:
    """Free text -> structured symptom list."""
    response = chat(
        model=sft_model,
        messages=[{
            "role": "user",
            "content": (
                "Extract the patient's symptoms as a list of short strings.\n"
                f"Patient: {patient_input}"
            ),
        }],
        format=Symptoms.model_json_schema(),
        options={"temperature": 0},
    )
    return Symptoms.model_validate_json(response.message.content)


parse_symptoms("I'm feeling not good today, I am coughing and sneezing")

In [ ]:
def lookup_symptoms(symptoms: list[str]) -> tuple[list, str, str]:
    """Ask the model to search, run the tool, append the tool result.

    Returns:
        conversation_history (with assistant tool call + tool message),
        the query the model chose,
        the tool result string.
    """
    conversation_history = [{
        "role": "user",
        "content": (
            f"The patient reported these symptoms: {symptoms}. "
            "What is the likely diagnosis?"
        ),
    }]
    response = chat(
        model=sft_model,
        messages=conversation_history,
        tools=[search_web],
        options={"temperature": 0},
    )
    conversation_history.append(response.message)

    tool_call = response.message.tool_calls[0].model_dump()["function"]
    tool_function = available_tool_dict[tool_call["name"]]
    tool_args = tool_call["arguments"]
    tool_result = tool_function(**tool_args)
    query = tool_args.get("query", "")

    conversation_history.append({"role": "tool", "content": tool_result})
    return conversation_history, query, tool_result


history, query, results = lookup_symptoms(["coughing", "sneezing"])
print("query:", query)
print(results)

In [ ]:
def diagnose(conversation_history: list) -> Diagnosis:
    """Structured diagnosis from the conversation (must already include tool results)."""
    response = chat(
        model=sft_model,
        messages=conversation_history,
        format=Diagnosis.model_json_schema(),
        options={"temperature": 0},
    )
    return Diagnosis.model_validate_json(response.message.content)


diagnose(history)

In [ ]:
def save_record(
    patient_input: str,
    symptoms: list[str],
    diagnosis: Diagnosis,
    search_query: str | None = None,
    search_results: str | None = None,
    path: str = "records.jsonl",
) -> Record:
    """Persist one intake record. No LLM needed."""
    record = Record(
        patient_input=patient_input,
        symptoms=symptoms,
        search_query=search_query,
        search_results=search_results,
        diagnosis=diagnosis,
    )
    with open(path, "a") as f:
        f.write(record.model_dump_json() + "\n")
    return record

In [ ]:
def respond(
    patient_input: str,
    diagnosis: Diagnosis,
    search_results: str,
) -> str:
    """Fill a template and reply to the patient (fresh messages, not the tool history)."""
    reply = chat(
        model=sft_model,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a friendly health intake assistant speaking directly to the patient. "
                    "Be brief. You are not a doctor; suggest seeing a clinician if symptoms "
                    "are severe or persist. Mention that their details have been recorded."
                ),
            },
            {
                "role": "user",
                "content": (
                    f"Patient said: {patient_input}\n"
                    f"Likely diagnosis: {diagnosis.diagnosis} "
                    f"(confidence: {diagnosis.confidence})\n"
                    f"Search results:\n{search_results}\n"
                    "Write a short reply to the patient."
                ),
            },
        ],
        options={"temperature": 0},
    )
    return reply.message.content

### Wrap it: `healthbot`

Once the steps are functions, the end-to-end pipeline is just composition — this is the diagram as code.

In [ ]:
def healthbot_fn(patient_input: str) -> str:
    parsed = parse_symptoms(patient_input)
    history, query, tool_result = lookup_symptoms(parsed.symptoms)
    diagnosis = diagnose(history)
    save_record(
        patient_input=patient_input,
        symptoms=parsed.symptoms,
        diagnosis=diagnosis,
        search_query=query,
        search_results=tool_result,
    )
    return respond(patient_input, diagnosis, tool_result)


print(healthbot_fn("I'm feeling not good today, I am coughing and sneezing"))

In [ ]:
print(healthbot_fn(
    "I've had a pounding headache behind my eyes for two days, "
    "bright light hurts, and I feel nauseous."
))